In [3]:
from supabase import create_client
from dotenv import load_dotenv
import os

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

response = supabase.storage.create_bucket(
    "user-pdfs",
    options={
        "public": False,
        "allowed_mime_types": ["application/pdf"],
        "file_size_limit": 10485760  # 10MB
    }
)

print(response)

{'name': 'user-pdfs'}


In [ ]:


def upload_pdf(file, user_id, session_id):
    file_path = f"{user_id}/{session_id}/{file.filename}"

    response = supabase.storage.from_("user-pdfs").upload(
        path=file_path,
        file=file,
        file_options={
            "content-type": "application/pdf",
            "upsert": "true"
        }
    )

    return file_path

In [5]:
def get_user_pdfs(user_id, session_id):
    path = f"{user_id}/{session_id}"

    response = supabase.storage.from_("user-pdfs").list(path)

    return response

In [6]:
def download_pdf(user_id, session_id, filename):
    path = f"{user_id}/{session_id}/{filename}"

    file = supabase.storage.from_("user-pdfs").download(path)

    return file

In [7]:


# Sample IDs
user_id = "user123"
session_id = "session456"

file_path = "./inputSQL.pdf"
file_name = "inputSQL.pdf"

# Storage path inside bucket
storage_path = f"{user_id}/{session_id}/{file_name}"

with open(file_path, "rb") as f:
    response = supabase.storage.from_("user-pdfs").upload(
        path=storage_path,
        file=f,
        file_options={
            "content-type": "application/pdf",
            "upsert": "true"
        }
    )

print("Uploaded to:", storage_path)
print(response)

Uploaded to: user123/session456/inputSQL.pdf
UploadResponse(path='user123/session456/inputSQL.pdf', full_path='user-pdfs/user123/session456/inputSQL.pdf', fullPath='user-pdfs/user123/session456/inputSQL.pdf')


In [8]:
files = supabase.storage.from_("user-pdfs").list(f"{user_id}/{session_id}")
print(files)

[{'name': 'inputSQL.pdf', 'id': '917e9de0-9347-46fd-9e3f-4c2762bc637e', 'updated_at': '2026-04-04T17:35:15.248Z', 'created_at': '2026-04-04T17:35:15.248Z', 'last_accessed_at': '2026-04-04T17:35:15.248Z', 'metadata': {'eTag': '"9cf52dadd65e44ebe3368f64d21bf30b"', 'size': 149693, 'mimetype': 'application/pdf', 'cacheControl': 'no-cache', 'lastModified': '2026-04-04T17:35:16.000Z', 'contentLength': 149693, 'httpStatusCode': 200}}]


In [9]:
file_bytes = supabase.storage.from_("user-pdfs").download(storage_path)

# Save locally
with open("downloaded_mypdf.pdf", "wb") as f:
    f.write(file_bytes)

print("Downloaded successfully ✅")

Downloaded successfully ✅
